In [ ]:
!git clone -b expand https://github.com/nnminh322/Continual.git
%cd /kaggle/working/Continual/expand_method/hypothesis_testing
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers>=4.30.0 numpy>=1.24.0 Pillow>=9.0.0 sentence-transformers>=2.2.0 tqdm>=4.65.0 scikit-learn>=1.3.0 shortuuid>=1.0.0
!pip install -e .

# 4. Verify
!python -c "from srt_router.pooled_mahalanobis import PooledMahalanobisRouter; print('✓ GPU pipeline ready')"

In [6]:
!git pull origin expand

From https://github.com/nnminh322/Continual
 * branch            expand     -> FETCH_HEAD
Already up to date.


In [2]:
!python scripts/generate_ins_emb.py \
    --model sentence-transformers/all-MiniLM-L6-v2 \
    --output ins_emb_single.pkl

Encoding 7 instructions with sentence-transformers/all-MiniLM-L6-v2...
tokenizer_config.json: 100%|███████████████████| 350/350 [00:00<00:00, 2.84MB/s]
vocab.txt: 232kB [00:00, 25.8MB/s]
tokenizer.json: 466kB [00:00, 61.0MB/s]
special_tokens_map.json: 100%|██████████████████| 112/112 [00:00<00:00, 602kB/s]
model.safetensors: 100%|███████████████████| 90.9M/90.9M [00:00<00:00, 98.2MB/s]
Loading weights: 100%|█| 103/103 [00:00<00:00, 1892.94it/s, Materializing param=
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  Shape: torch.Size([7, 384])
  Saved to ins_emb_single.pkl
  Also saved numpy to ins_emb_single.npy


In [ ]:
!python experiments/smolora/if_router/routing_accuracy.py \
    --ins_emb ins_emb_single.pkl \
    --task_order ScienceQA TextVQA GQA VQAv2 Flickr30k ImageNet Place365 \
    --shrinkage ridge \
    --device cuda \
    --n_samples 200 \
    --noise_std 0.1

In [ ]:
# VU Router: CLIP image features → SRT Mahalanobis vs Cosine.
# Skips gracefully when images are missing (prints warning, skips task).
!python experiments/smolora/vu_router/routing_accuracy.py \
    --data_root /data/zqwang/moe_cl_data/dataset \
    --task_names ScienceQA TextVQA GQA VQAv2 \
    --clip_model openai/clip-vit-large-patch14-336 \
    --shrinkage ridge \
    --device cuda \
    --n_train 500 \
    --n_test 200

In [ ]:
# Dual Router: VU (CLIP) + IF (instruction embedding) combined.
# Skips gracefully when images are missing (uses --data_root=synthetic to force synthetic mode).
!python experiments/smolora/dual_router/routing_accuracy.py \
    --ins_emb ins_emb_single.pkl \
    --data_root /data/zqwang/moe_cl_data/dataset \
    --task_names ScienceQA TextVQA GQA VQAv2 \
    --clip_model openai/clip-vit-large-patch14-336 \
    --alpha 0.5 \
    --shrinkage ridge \
    --device cuda

In [ ]:
# HiDe-LLaVA: CLIP features → SRT Mahalanobis vs Cosine.
# Fails gracefully when no images found (prints diagnostic, exits cleanly).
!python experiments/hide/cosine_router/routing_accuracy.py \
    --data_root /data/zqwang/moe_cl_data/dataset \
    --task_names ScienceQA TextVQA GQA VQAv2 VizWiz TextCaps \
    --clip_model openai/clip-vit-large-patch14-336 \
    --shrinkage ridge \
    --device cuda

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Option B: End-to-End (requires trained SMoLoRA checkpoint)
# ───────────────────────────────────────────────────────────────────
# BEFORE RUNNING: Set SMOLORA_REPO to your SMoLoRA directory.
# Example on server:
#   SMOLORA_REPO=/data/zqwang/moe_cl_data/SMoLoRA
#   MODEL_PATH=/path/to/your/trained/smolora/checkpoint
#   MODEL_BASE=/path/to/vicuna-7b-v1.5
# ───────────────────────────────────────────────────────────────────
# If paths are missing → prints clear diagnostic message, no crash.
!export SMOLORA_REPO=/data/zqwang/moe_cl_data/SMoLoRA
!python experiments/smolora/if_router/end_to_end.py \
    --model_path /data/zqwang/moe_cl_data/smolora_checkpoints/smolora_if \
    --model_base /data/zqwang/moe_cl_data/vicuna-7b-v1.5 \
    --ins_emb ins_emb_single.pkl \
    --task_order ScienceQA TextVQA GQA VQAv2 \
    --routing_mode all \
    --scoring_func vqav2 \
    --device cuda \
    --output_dir results_smolora_if_e2e

In [ ]:
# VU Router End-to-End: CLIP features → SRT routing on trained SMoLoRA.
# Same pre-reqs as IF end-to-end (trained checkpoint + SMOLORA_REPO).
!export SMOLORA_REPO=/data/zqwang/moe_cl_data/SMoLoRA
!python experiments/smolora/vu_router/end_to_end.py \
    --model_path /data/zqwang/moe_cl_data/smolora_checkpoints/smolora_vu \
    --model_base /data/zqwang/moe_cl_data/vicuna-7b-v1.5 \
    --clip_model openai/clip-vit-large-patch14-336 \
    --task_images_root /data/zqwang/moe_cl_data/dataset \
    --task_order ScienceQA TextVQA GQA VQAv2 \
    --routing_mode all \
    --device cuda

In [9]:
!export SMOLORA_DATA=/data/zqwang/moe_cl_data/dataset
!export HIDE_DATA=/your_path/datasets


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Batch Run — all Option A experiments + Option B placeholder
# ───────────────────────────────────────────────────────────────────
# ── Setup ──────────────────────────────────────────────────────────
!python scripts/generate_ins_emb.py --output ins_emb_single.pkl

# ── Option A: Routing Accuracy (no checkpoint needed) ───────────────

# 1) IF Router (fastest, no images needed)
!python experiments/smolora/if_router/routing_accuracy.py \
    --ins_emb ins_emb_single.pkl \
    --task_order ScienceQA TextVQA GQA VQAv2 \
    --shrinkage ridge --device cpu

# 2) VU Router (needs images — skips gracefully if missing)
!python experiments/smolora/vu_router/routing_accuracy.py \
    --data_root /data/zqwang/moe_cl_data/dataset \
    --task_names ScienceQA TextVQA GQA VQAv2 \
    --clip_model openai/clip-vit-large-patch14-336 \
    --shrinkage ridge --device cuda

# 3) Dual Router (VU + IF, skips gracefully)
!python experiments/smolora/dual_router/routing_accuracy.py \
    --ins_emb ins_emb_single.pkl \
    --data_root /data/zqwang/moe_cl_data/dataset \
    --task_names ScienceQA TextVQA GQA VQAv2 \
    --clip_model openai/clip-vit-large-patch14-336 \
    --alpha 0.5 --device cuda

# 4) HiDe Router (6 datasets, fails gracefully if no images)
!python experiments/hide/cosine_router/routing_accuracy.py \
    --data_root /data/zqwang/moe_cl_data/dataset \
    --task_names ScienceQA TextVQA GQA VQAv2 VizWiz TextCaps \
    --clip_model openai/clip-vit-large-patch14-336 \
    --device cuda

# ── Option B: End-to-End (requires trained checkpoint + SMOLORA_REPO) ──
# Uncomment and fill in your paths after training SMoLoRA:
# !export SMOLORA_REPO=/data/zqwang/moe_cl_data/SMoLoRA
# !python experiments/smolora/if_router/end_to_end.py \
#     --model_path /path/to/checkpoint \
#     --model_base /path/to/vicuna-7b \
#     --ins_emb ins_emb_single.pkl \
#     --task_order ScienceQA TextVQA GQA VQAv2 \
#     --routing_mode all --scoring_func vqav2